# 1. Dataset Description

This section provides a comprehensive overview of the dataset used in this analysis, including its source, structure, variable types, target variables, and known limitations.

## 1.1 Data Source

- **Platform:** [Kaggle](https://www.kaggle.com)
- **Dataset name:** Instagram Users Lifestyle Dataset
- **Preferred file:** `instagram_users_lifestyle.csv`
- **Fallback file for shared GitHub clones:** `instagram_users_lifestyle_sample_10000_engineered.csv`

The project was originally developed on the full synthetic Instagram dataset. For portability, this notebook can also run from the bundled 10,000-row engineered sample when the full CSV is not available locally.

In [1]:
from pathlib import Path
import pandas as pd

dataset_candidates = [
    Path("instagram_users_lifestyle.csv"),
    Path("instagram_users_lifestyle_sample_10000_engineered.csv"),
]

dataset_path = next((path for path in dataset_candidates if path.exists()), None)
if dataset_path is None:
    raise FileNotFoundError(
        "No dataset file was found. Add 'instagram_users_lifestyle.csv' to the project root "
        "or keep 'instagram_users_lifestyle_sample_10000_engineered.csv' beside the notebook."
    )

df = pd.read_csv(dataset_path)
print(f"Dataset loaded successfully from: {dataset_path.name}")
print(f"Shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
df.head(3)

Dataset loaded successfully from: instagram_users_lifestyle.csv
Shape: 1,547,896 rows x 58 columns


,user_id,app_name,age,gender,country,urban_rural,income_level,employment_status,education_level,relationship_status,...,last_login_date,average_session_length_minutes,content_type_preference,preferred_content_theme,privacy_setting_level,two_factor_auth_enabled,biometric_login_used,linked_accounts_count,subscription_status,user_engagement_score
0,1,Instagram,51,Female,India,Rural,High,Retired,Bachelor’s,Single,...,2025-11-02,5.0,Mixed,Tech,Private,Yes,No,0,Free,7.83
1,2,Instagram,64,Female,United Kingdom,Urban,Middle,Full-time employed,Other,Divorced,...,2025-03-22,14.8,Photos,Fashion,Public,No,No,3,Free,1.43
2,3,Instagram,41,Female,Canada,Urban,Middle,Student,Bachelor’s,In a relationship,...,2025-08-10,5.0,Mixed,Other,Public,Yes,Yes,1,Free,9.67


## 1.2 Number of Samples and Features

In [ ]:
n_samples, n_features = df.shape
print(f"Number of samples  : {n_samples:,}")
print(f"Number of features : {n_features}")
print(f"  ├─ Input features (excl. targets & ID) : {n_features - 4}")
print(f"  └─ Target variables                    : 3  (self_reported_happiness, perceived_stress_score, user_engagement_score)")

## 1.3 Types of Variables

Below we classify each column into **Numerical** (continuous or discrete) or **Categorical** (nominal/ordinal) types.

In [ ]:
numerical_cols = df.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_cols = df.select_dtypes(include=["object"]).columns.tolist()

print(f"Numerical columns  ({len(numerical_cols)}):")
for c in numerical_cols:
    print(f"  - {c}  [{df[c].dtype}]")

print(f"\nCategorical columns ({len(categorical_cols)}):")
for c in categorical_cols:
    unique_vals = df[c].nunique()
    print(f"  - {c}  (unique values: {unique_vals})")

## 1.4 Target Variables

Three target variables are considered for prediction tasks in this dataset:

| Variable | Type | Scale | Description |
|---|---|---|---|
| `self_reported_happiness` | Numerical (discrete) | 1 – 10 | User's subjective happiness rating |
| `perceived_stress_score` | Numerical (discrete) | 1 – 20 | Self-assessed stress level (higher = more stressed) |
| `user_engagement_score` | Numerical (continuous) | 0.0 – ~10.0 | Composite score of platform engagement activity |

In [ ]:
targets = ["self_reported_happiness", "perceived_stress_score", "user_engagement_score"]
print("Target variable statistics:\n")
print(df[targets].describe().round(2).to_string())

## 1.5 Known Limitations

- **Fully synthetic data:** The dataset was artificially generated; it does not represent real user behaviour and should not be used for real-world inference.
- **Demographic inconsistencies:** The generation process contains illogical combinations, for example users aged 15 who already have children, or very young users with employment statuses that are unrealistic for their age.
- **`perceived_stress_score` scale mismatch:** The output statistics show values ranging from 0 to 40, which exceeds the standard PSS-10 (0–40) scale but is inconsistent with the column description that implies a 1–20 range. Values should be verified before use.
- **Single platform:** All records belong to Instagram only (`app_name` has 1 unique value), limiting cross-platform generalisation.

## 1.6 Feature Categorization

The 58 columns are grouped into **6 logical categories** based on the nature of the information they capture:

| Category | Description |
|---|---|
| **Demographics** | Basic user identity and socioeconomic background |
| **Lifestyle & Health** | Physical health indicators and daily habits |
| **Social & Personal Life** | Off-platform social activities and personal interests |
| **Instagram Behaviour** | In-app engagement and content consumption metrics |
| **Account & Settings** | Account configuration and platform preferences |
| **Target Variables** | Outcome variables for prediction tasks |

In [ ]:
feature_groups = {
    "Demographics": [
        "user_id", "app_name", "age", "gender", "country",
        "urban_rural", "income_level", "employment_status",
        "education_level", "relationship_status", "has_children"
    ],
    "Lifestyle & Health": [
        "exercise_hours_per_week", "sleep_hours_per_night", "diet_quality",
        "smoking", "alcohol_frequency", "body_mass_index",
        "blood_pressure_systolic", "blood_pressure_diastolic", "daily_steps_count"
    ],
    "Social & Personal Life": [
        "weekly_work_hours", "hobbies_count", "social_events_per_month",
        "books_read_per_year", "volunteer_hours_per_month", "travel_frequency_per_year"
    ],
    "Instagram Behaviour": [
        "daily_active_minutes_instagram", "sessions_per_day", "posts_created_per_week",
        "reels_watched_per_day", "stories_viewed_per_day", "likes_given_per_day",
        "comments_written_per_day", "dms_sent_per_week", "dms_received_per_week",
        "ads_viewed_per_day", "ads_clicked_per_day", "time_on_feed_per_day",
        "time_on_explore_per_day", "time_on_messages_per_day", "time_on_reels_per_day",
        "followers_count", "following_count", "notification_response_rate",
        "average_session_length_minutes", "content_type_preference", "preferred_content_theme"
    ],
    "Account & Settings": [
        "uses_premium_features", "account_creation_year", "last_login_date",
        "privacy_setting_level", "two_factor_auth_enabled", "biometric_login_used",
        "linked_accounts_count", "subscription_status"
    ],
    "Target Variables": [
        "self_reported_happiness", "perceived_stress_score", "user_engagement_score"
    ],
}

print(f"{'Category':<25} {'Count':>5}  Columns")
print("-" * 90)
for group, cols in feature_groups.items():
    dtypes = [str(df[c].dtype) for c in cols]
    cols_preview = ", ".join(cols[:4]) + (f", ... (+{len(cols)-4} more)" if len(cols) > 4 else "")
    print(f"{group:<25} {len(cols):>5}  {cols_preview}")

print("\nTotal columns accounted for:", sum(len(v) for v in feature_groups.values()))

In [ ]:
print(f"{'Category':<25} {'Total':>5}  {'Numerical':>9}  {'Categorical':>11}")
print("-" * 60)
for group, cols in feature_groups.items():
    num = sum(1 for c in cols if df[c].dtype in ["int64", "float64"])
    cat = len(cols) - num
    print(f"{group:<25} {len(cols):>5}  {num:>9}  {cat:>11}")

# 2. Target Variable Analysis

This section examines the three key outcome variables in depth:

- **`self_reported_happiness`** — Subjective happiness score (1–10, discrete)
- **`perceived_stress_score`** — Self-assessed stress level (0–40, discrete)
- **`user_engagement_score`** — Composite Instagram engagement score (continuous)

For each target we inspect the **distribution shape**, **central tendency**, **spread**, and **value frequencies**. Finally, we measure the **pairwise correlations** among the three targets.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

targets = ["self_reported_happiness", "perceived_stress_score", "user_engagement_score"]
titles  = ["Self-Reported Happiness (1–10)", "Perceived Stress Score (0–40)", "User Engagement Score"]
colors  = ["#4C72B0", "#DD8452", "#55A868"]

fig, axes = plt.subplots(3, 2, figsize=(16, 15))
fig.suptitle("Target Variable Analysis", fontsize=16, fontweight="bold", y=1.01)

for i, (col, title, color) in enumerate(zip(targets, titles, colors)):
    series = df[col]

    # --- Distribution plot ---
    ax_dist = axes[i, 0]
    if series.nunique() <= 20:           # discrete — bar chart
        counts = series.value_counts().sort_index()
        ax_dist.bar(counts.index, counts.values, color=color, edgecolor="white", width=0.7)
        ax_dist.set_xlabel(col)
        ax_dist.set_ylabel("Count")
    else:                                # continuous — histogram
        ax_dist.hist(series.dropna(), bins=60, color=color, edgecolor="white")
        ax_dist.set_xlabel(col)
        ax_dist.set_ylabel("Count")
    ax_dist.set_title(f"{title}\nDistribution")
    ax_dist.axvline(series.mean(),   color="red",    linestyle="--", linewidth=1.5, label=f"Mean  {series.mean():.2f}")
    ax_dist.axvline(series.median(), color="black",  linestyle=":",  linewidth=1.5, label=f"Median {series.median():.2f}")
    ax_dist.legend(fontsize=8)

    # --- Stats summary ---
    ax_stat = axes[i, 1]
    ax_stat.axis("off")
    stats = series.describe()
    summary = (
        f"  Count   : {int(stats['count']):>12,}\n"
        f"  Mean    : {stats['mean']:>12.3f}\n"
        f"  Median  : {series.median():>12.3f}\n"
        f"  Std     : {stats['std']:>12.3f}\n"
        f"  Min     : {stats['min']:>12.3f}\n"
        f"  25%     : {stats['25%']:>12.3f}\n"
        f"  75%     : {stats['75%']:>12.3f}\n"
        f"  Max     : {stats['max']:>12.3f}\n"
        f"  Skewness: {series.skew():>12.3f}\n"
        f"  Kurtosis: {series.kurt():>12.3f}\n"
        f"  Missing : {series.isna().sum():>12,}"
    )
    ax_stat.text(0.05, 0.95, f"Summary — {title}", transform=ax_stat.transAxes,
                 fontsize=11, fontweight="bold", va="top")
    ax_stat.text(0.05, 0.82, summary, transform=ax_stat.transAxes,
                 fontsize=10, va="top", fontfamily="monospace",
                 bbox=dict(boxstyle="round,pad=0.5", facecolor=color, alpha=0.15))

plt.tight_layout()
plt.show()

# --- Pairwise correlation ---
print("\nPairwise Pearson correlation between target variables:")
print(df[targets].corr().round(4).to_string())

# 3. Exploratory Data Analysis (EDA)

We use a **50 % random sample** of the dataset for all visualisations in this section to keep rendering fast while remaining statistically representative.

Subsections covered:
1. **Demographic Overview** — age, gender, country, generation
2. **Socioeconomic Factors** — income, employment, education, relationship status
3. **Instagram Usage Patterns** — daily minutes, sessions, time by activity, content preferences
4. **Usage by Generation** — how different cohorts engage with the platform

In [ ]:
import pandas as pd
import plotly.io as pio
import plotly.graph_objects as go
from plotly.subplots import make_subplots


# Render Plotly figures in the browser so large interactive outputs are not embedded in the notebook file.
pio.renderers.default = 'browser'

# ---------- colour palette ----------
COLORS = {
    'primary':   '#4C72B0',
    'secondary': '#DD8452',
    'accent':    '#8172B2',
    'success':   '#55A868',
    'info':      '#64B5CD',
    'warning':   '#CCB974',
}
PALETTE = list(COLORS.values())

# ---------- 50 % sample ----------
df_sample = df.sample(frac=0.5, random_state=42)
print(f"Full dataset : {len(df):,} rows")
print(f"Sample (50%) : {len(df_sample):,} rows")

# ---------- generation labels ----------
GEN_ORDER = ['Gen Alpha', 'Gen Z', 'Millennial', 'Gen X', 'Boomer+']

def assign_generation(age):
    if age < 12:  return 'Gen Alpha'
    elif age < 28: return 'Gen Z'
    elif age < 44: return 'Millennial'
    elif age < 60: return 'Gen X'
    else:          return 'Boomer+'

df['generation']        = df['age'].apply(assign_generation)
df_sample['generation'] = df_sample['age'].apply(assign_generation)
print("Generation column created.")

## 3.1 Demographic Overview

In [ ]:
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=['Age Distribution', 'Gender Distribution',
                    'Country Distribution', 'Generation Distribution'],
    specs=[[{'type': 'histogram'}, {'type': 'pie'}],
           [{'type': 'bar'},       {'type': 'bar'}]]
)

# Age histogram
fig.add_trace(go.Histogram(x=df_sample['age'], marker_color=COLORS['primary'],
                           nbinsx=50, name='Age'), row=1, col=1)

# Gender pie
gender_counts = df_sample['gender'].value_counts()
fig.add_trace(go.Pie(labels=gender_counts.index, values=gender_counts.values,
                     marker_colors=PALETTE), row=1, col=2)

# Country bar
country_counts = df_sample['country'].value_counts().head(10)
fig.add_trace(go.Bar(x=country_counts.index, y=country_counts.values,
                     marker_color=COLORS['secondary'], name='Country'), row=2, col=1)

# Generation bar
gen_counts = df_sample['generation'].value_counts().reindex(GEN_ORDER)
fig.add_trace(go.Bar(x=gen_counts.index, y=gen_counts.values,
                     marker_color=COLORS['accent'], name='Generation'), row=2, col=2)

fig.update_layout(height=700, showlegend=False, title_text='Demographic Overview')
fig.show()

## 3.2 Socioeconomic Factors

In [ ]:
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=['Income Level', 'Employment Status',
                    'Education Level', 'Relationship Status']
)

income_counts = df_sample['income_level'].value_counts()
fig.add_trace(go.Bar(x=income_counts.index, y=income_counts.values,
                     marker_color=COLORS['success'], name='Income'), row=1, col=1)

emp_counts = df_sample['employment_status'].value_counts()
fig.add_trace(go.Bar(x=emp_counts.index, y=emp_counts.values,
                     marker_color=COLORS['info'], name='Employment'), row=1, col=2)

edu_counts = df_sample['education_level'].value_counts()
fig.add_trace(go.Bar(x=edu_counts.index, y=edu_counts.values,
                     marker_color=COLORS['warning'], name='Education'), row=2, col=1)

rel_counts = df_sample['relationship_status'].value_counts()
fig.add_trace(go.Bar(x=rel_counts.index, y=rel_counts.values,
                     marker_color=COLORS['primary'], name='Relationship'), row=2, col=2)

fig.update_layout(height=600, showlegend=False, title_text='Socioeconomic Demographics')
fig.update_xaxes(tickangle=45)
fig.show()

## 3.3 Instagram Usage Patterns

In [ ]:
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=['Daily Active Minutes', 'Sessions per Day',
                    'Avg Time by Activity Type', 'Content Type Preference']
)

fig.add_trace(go.Histogram(x=df_sample['daily_active_minutes_instagram'],
                           marker_color=COLORS['primary'], nbinsx=50, name='Daily Mins'), row=1, col=1)

fig.add_trace(go.Histogram(x=df_sample['sessions_per_day'],
                           marker_color=COLORS['secondary'], nbinsx=30, name='Sessions'), row=1, col=2)

time_cols   = ['time_on_feed_per_day', 'time_on_explore_per_day',
               'time_on_messages_per_day', 'time_on_reels_per_day']
time_labels = ['Feed', 'Explore', 'Messages', 'Reels']
time_means  = df_sample[time_cols].mean()
fig.add_trace(go.Bar(x=time_labels, y=time_means.values,
                     marker_color=[COLORS['primary'], COLORS['secondary'],
                                   COLORS['success'], COLORS['accent']],
                     name='Avg Time'), row=2, col=1)

content_counts = df_sample['content_type_preference'].value_counts()
fig.add_trace(go.Bar(x=content_counts.index, y=content_counts.values,
                     marker_color=COLORS['warning'], name='Content'), row=2, col=2)

fig.update_layout(height=600, showlegend=False, title_text='Instagram Usage Patterns')
fig.show()

print("\nInstagram Usage Statistics:")
print(f"  Avg daily active minutes : {df['daily_active_minutes_instagram'].mean():.1f}")
print(f"  Median daily active mins : {df['daily_active_minutes_instagram'].median():.1f}")
print(f"  Avg sessions per day     : {df['sessions_per_day'].mean():.1f}")
print(f"  Avg reels watched/day    : {df['reels_watched_per_day'].mean():.1f}")
print(f"  Avg stories viewed/day   : {df['stories_viewed_per_day'].mean():.1f}")

## 3.4 Usage by Generation

In [ ]:
usage_by_gen = df.groupby('generation').agg(
    daily_active_minutes = ('daily_active_minutes_instagram', 'mean'),
    reels_per_day        = ('reels_watched_per_day',          'mean'),
    posts_per_week       = ('posts_created_per_week',         'mean'),
    stories_per_day      = ('stories_viewed_per_day',         'mean'),
    sessions_per_day     = ('sessions_per_day',               'mean'),
).reindex(GEN_ORDER).round(1)

print("Usage Metrics by Generation:")
display(usage_by_gen)

# Grouped bar chart
metrics      = ['daily_active_minutes', 'reels_per_day', 'stories_per_day']
metric_names = ['Daily Minutes', 'Reels/Day', 'Stories/Day']
bar_colors   = [COLORS['primary'], COLORS['secondary'], COLORS['accent']]

fig = go.Figure()
for metric, name, color in zip(metrics, metric_names, bar_colors):
    fig.add_trace(go.Bar(name=name, x=GEN_ORDER,
                         y=usage_by_gen[metric].values,
                         marker_color=color))

fig.update_layout(
    barmode='group',
    title='Instagram Usage by Generation',
    xaxis_title='Generation',
    yaxis_title='Average Value',
    height=450,
    legend_title='Metric',
)
fig.show()

# 4. Correlation Analysis

We examine linear relationships at two levels:

1. **Feature–Feature correlations** — a full Pearson heatmap of all numerical columns to spot multicollinearity and natural clusters.
2. **Feature–Target correlations** — for each of the three target variables (`self_reported_happiness`, `perceived_stress_score`, `user_engagement_score`) we rank every numerical feature by its absolute Pearson correlation and visualise the **top 15 strongest associations**.

> All correlations are computed on the full dataset (1.5 M rows) for maximum statistical reliability.

## 4.1 Full Numerical Correlation Heatmap

In [ ]:
import plotly.graph_objects as go
import numpy as np

# Drop non-informative columns before computing correlations
drop_cols = ['user_id', 'account_creation_year']
num_df = df.select_dtypes(include='number').drop(columns=drop_cols, errors='ignore')

corr = num_df.corr()
labels = corr.columns.tolist()

# Mask upper triangle for a cleaner look
mask = np.triu(np.ones(corr.shape, dtype=bool), k=1)
corr_display = corr.copy()
corr_display[mask] = None

fig = go.Figure(go.Heatmap(
    z=corr_display.values,
    x=labels,
    y=labels,
    colorscale='RdBu',
    zmid=0,
    zmin=-1, zmax=1,
    colorbar=dict(title='Pearson r'),
    hoverongaps=False,
    text=corr_display.round(2).values,
    texttemplate='%{text}',
    textfont=dict(size=7),
))

fig.update_layout(
    title='Pearson Correlation Matrix — All Numerical Features',
    height=900,
    width=1000,
    xaxis=dict(tickangle=45, tickfont=dict(size=9)),
    yaxis=dict(tickfont=dict(size=9)),
)
fig.show()

## 4.2 Feature–Target Correlations

Top 15 numerical features most correlated with each target variable (sorted by absolute Pearson r).

In [ ]:
from plotly.subplots import make_subplots

TARGETS = ['self_reported_happiness', 'perceived_stress_score', 'user_engagement_score']
TARGET_LABELS = ['Self-Reported Happiness', 'Perceived Stress Score', 'User Engagement Score']
TARGET_COLORS = [COLORS['success'], COLORS['secondary'], COLORS['primary']]
TOP_N = 15

# Feature pool: all numerics except targets and ID columns
feature_pool = num_df.drop(columns=TARGETS, errors='ignore')

fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=TARGET_LABELS,
    horizontal_spacing=0.12,
)

for col_idx, (target, label, color) in enumerate(zip(TARGETS, TARGET_LABELS, TARGET_COLORS), start=1):
    corr_series = feature_pool.corrwith(df[target]).dropna()
    top = corr_series.reindex(corr_series.abs().nlargest(TOP_N).index).sort_values()

    bar_colors = [COLORS['info'] if v >= 0 else COLORS['secondary'] for v in top.values]

    fig.add_trace(
        go.Bar(
            x=top.values,
            y=top.index,
            orientation='h',
            marker_color=bar_colors,
            name=label,
            text=top.round(3).values,
            textposition='outside',
            textfont=dict(size=9),
        ),
        row=1, col=col_idx,
    )
    fig.update_xaxes(title_text='Pearson r', row=1, col=col_idx, range=[-1, 1])

fig.update_layout(
    title='Top 15 Features Correlated with Each Target Variable',
    height=600,
    showlegend=False,
    margin=dict(l=200),
)
fig.show()

In [ ]:
import pandas as pd

print("=" * 65)
for target, label in zip(TARGETS, TARGET_LABELS):
    corr_series = feature_pool.corrwith(df[target]).dropna()
    top = corr_series.reindex(corr_series.abs().nlargest(TOP_N).index).sort_values(key=abs, ascending=False)
    top_df = pd.DataFrame({'Feature': top.index, 'Pearson r': top.round(4).values})
    print(f"\n  {label}")
    print("-" * 65)
    print(top_df.to_string(index=False))
print("=" * 65)

## 4.4 Key Relationship Scatter Plots

Six scatter plots that put the strongest correlations under a visual microscope.  
Each plot uses a **5 000-row sub-sample** of the 50 % sample for fast, readable rendering.

In [ ]:
scatter_sample = df_sample.sample(5000, random_state=42)

fig = make_subplots(
    rows=2, cols=3,
    subplot_titles=[
        'Stress vs Daily Usage',
        'Happiness vs Daily Usage',
        'Age vs Reels Watched',
        'Exercise vs Happiness',
        'Sleep vs Stress',
        'Sessions vs Engagement',
    ],
)

# row 1 ─────────────────────────────────────────────────────────────
fig.add_trace(go.Scatter(
    x=scatter_sample['daily_active_minutes_instagram'],
    y=scatter_sample['perceived_stress_score'],
    mode='markers', name='Stress vs Usage',
    marker=dict(size=3, color=COLORS['secondary'], opacity=0.5),
), row=1, col=1)
fig.update_xaxes(title_text='Daily Active Minutes', row=1, col=1)
fig.update_yaxes(title_text='Perceived Stress Score', row=1, col=1)

fig.add_trace(go.Scatter(
    x=scatter_sample['daily_active_minutes_instagram'],
    y=scatter_sample['self_reported_happiness'],
    mode='markers', name='Happiness vs Usage',
    marker=dict(size=3, color=COLORS['success'], opacity=0.5),
), row=1, col=2)
fig.update_xaxes(title_text='Daily Active Minutes', row=1, col=2)
fig.update_yaxes(title_text='Self-Reported Happiness', row=1, col=2)

fig.add_trace(go.Scatter(
    x=scatter_sample['age'],
    y=scatter_sample['reels_watched_per_day'],
    mode='markers', name='Age vs Reels',
    marker=dict(size=3, color=COLORS['primary'], opacity=0.5),
), row=1, col=3)
fig.update_xaxes(title_text='Age', row=1, col=3)
fig.update_yaxes(title_text='Reels Watched / Day', row=1, col=3)

# row 2 ─────────────────────────────────────────────────────────────
fig.add_trace(go.Scatter(
    x=scatter_sample['exercise_hours_per_week'],
    y=scatter_sample['self_reported_happiness'],
    mode='markers', name='Exercise vs Happiness',
    marker=dict(size=3, color=COLORS['success'], opacity=0.5),
), row=2, col=1)
fig.update_xaxes(title_text='Exercise Hours / Week', row=2, col=1)
fig.update_yaxes(title_text='Self-Reported Happiness', row=2, col=1)

fig.add_trace(go.Scatter(
    x=scatter_sample['sleep_hours_per_night'],
    y=scatter_sample['perceived_stress_score'],
    mode='markers', name='Sleep vs Stress',
    marker=dict(size=3, color=COLORS['info'], opacity=0.5),
), row=2, col=2)
fig.update_xaxes(title_text='Sleep Hours / Night', row=2, col=2)
fig.update_yaxes(title_text='Perceived Stress Score', row=2, col=2)

fig.add_trace(go.Scatter(
    x=scatter_sample['sessions_per_day'],
    y=scatter_sample['user_engagement_score'],
    mode='markers', name='Sessions vs Engagement',
    marker=dict(size=3, color=COLORS['accent'], opacity=0.5),
), row=2, col=3)
fig.update_xaxes(title_text='Sessions / Day', row=2, col=3)
fig.update_yaxes(title_text='User Engagement Score', row=2, col=3)

fig.update_layout(
    height=650,
    showlegend=False,
    title_text='Key Relationship Scatter Plots',
)
fig.show()

# 5. Lifestyle & Health Deep Dive

This section examines how **usage intensity**, **physical health**, and **dietary habits** relate to the three target variables.

Subsections:
1. **Well-being by Usage Category** — bucket daily active minutes and compare average stress and happiness across groups
2. **Health & Lifestyle Distributions** — BMI, steps, sleep, exercise, blood pressure, diet quality
3. **Diet Quality × Stress Level** — cross-tabulation heatmap

## 5.1 Well-being by Usage Category

In [ ]:
import plotly.express as px

# Create usage buckets
df_sample['usage_category'] = pd.cut(
    df_sample['daily_active_minutes_instagram'],
    bins=[0, 30, 60, 120, 180, 500],
    labels=['Light (<30m)', 'Moderate (30-60m)', 'Heavy (1-2h)', 'Very Heavy (2-3h)', 'Extreme (3h+)']
)

# Aggregate stats per bucket
stress_by_usage = df_sample.groupby('usage_category', observed=True).agg(
    Avg_Stress=('perceived_stress_score', 'mean'),
    Stress_Std=('perceived_stress_score', 'std'),
    Avg_Happiness=('self_reported_happiness', 'mean'),
    Happiness_Std=('self_reported_happiness', 'std'),
    Count=('perceived_stress_score', 'count'),
).round(2)

print("Well-being Metrics by Usage Category:")
display(stress_by_usage)

# Bar charts with error bars
usage_cats = stress_by_usage.index.tolist()

fig = make_subplots(rows=1, cols=2,
                    subplot_titles=['Average Stress by Usage', 'Average Happiness by Usage'])

fig.add_trace(go.Bar(
    x=usage_cats, y=stress_by_usage['Avg_Stress'],
    error_y=dict(type='data', array=(stress_by_usage['Stress_Std'] / 10).tolist()),
    marker_color=COLORS['secondary'], name='Stress',
), row=1, col=1)

fig.add_trace(go.Bar(
    x=usage_cats, y=stress_by_usage['Avg_Happiness'],
    error_y=dict(type='data', array=(stress_by_usage['Happiness_Std'] / 10).tolist()),
    marker_color=COLORS['success'], name='Happiness',
), row=1, col=2)

fig.update_layout(height=420, showlegend=False,
                  title_text='Well-being by Instagram Usage Intensity')
fig.update_xaxes(tickangle=45)
fig.show()

## 5.2 Health & Lifestyle Metrics Distribution

In [ ]:
fig = make_subplots(
    rows=2, cols=3,
    subplot_titles=['BMI Distribution', 'Daily Steps', 'Sleep Hours',
                    'Exercise Hours / Week', 'Blood Pressure (Systolic)', 'Diet Quality']
)

fig.add_trace(go.Histogram(x=df_sample['body_mass_index'],
                           marker_color=COLORS['info'], nbinsx=40, name='BMI'), row=1, col=1)

fig.add_trace(go.Histogram(x=df_sample['daily_steps_count'],
                           marker_color=COLORS['success'], nbinsx=40, name='Steps'), row=1, col=2)

fig.add_trace(go.Histogram(x=df_sample['sleep_hours_per_night'],
                           marker_color=COLORS['secondary'], nbinsx=30, name='Sleep'), row=1, col=3)

fig.add_trace(go.Histogram(x=df_sample['exercise_hours_per_week'],
                           marker_color=COLORS['primary'], nbinsx=30, name='Exercise'), row=2, col=1)

fig.add_trace(go.Histogram(x=df_sample['blood_pressure_systolic'],
                           marker_color=COLORS['accent'], nbinsx=40, name='BP'), row=2, col=2)

diet_counts = df_sample['diet_quality'].value_counts()
fig.add_trace(go.Bar(x=diet_counts.index, y=diet_counts.values,
                     marker_color=COLORS['warning'], name='Diet'), row=2, col=3)

fig.update_layout(height=600, showlegend=False,
                  title_text='Health & Lifestyle Metrics Distribution')
fig.show()

## 5.3 Diet Quality × Stress Level

In [ ]:
# Stress buckets
df_sample['stress_category'] = pd.cut(
    df_sample['perceived_stress_score'],
    bins=[0, 10, 20, 30, 50],
    labels=['Low', 'Moderate', 'High', 'Very High']
)

# Cross-tabulation (row-normalised %)
stress_diet = pd.crosstab(
    df_sample['diet_quality'],
    df_sample['stress_category'],
    normalize='index'
) * 100

print("Stress Level Distribution by Diet Quality (%):")
display(stress_diet.round(1))

# Heatmap
fig = px.imshow(
    stress_diet.round(1),
    labels=dict(x='Stress Level', y='Diet Quality', color='% of Row'),
    color_continuous_scale='RdYlGn_r',
    text_auto=True,
    title='Stress Level Distribution by Diet Quality',
)
fig.update_layout(height=420)
fig.show()

# Key Findings & EDA Summary

A consolidated summary of all insights uncovered across Sections 2–5.

---

## Section 2 — Target Variable Analysis

| Target | Range | Mean | Std | Shape |
|---|---|---|---|---|
| `self_reported_happiness` | 1 – 10 | 5.50 | 2.87 | Uniform — all values equally likely |
| `perceived_stress_score` | 0 – 40 | 19.99 | 11.83 | Uniform — actual scale is 0–40, not 1–20 as documented |
| `user_engagement_score` | 0.67 – 18.67 | 1.64 | 1.81 | Strongly right-skewed — most users cluster near 1 |

**Pairwise correlations between targets:**
- Happiness ↔ Stress: r ≈ 0.00 (no relationship)
- Stress ↔ Engagement: r = −0.44 (moderate negative)
- Happiness ↔ Engagement: r = +0.27 (mild positive)

---

## Section 3 — Exploratory Data Analysis

### 3.1 Demographics
- Age is broadly distributed across 13–90+; the dataset contains no Gen Alpha users (under 12).
- Gender split is roughly equal across Female, Male, Non-binary, and Other.
- 10 countries are represented; all with near-equal representation (synthetic uniform sampling).
- Millennials and Gen X are the largest generational cohorts.

### 3.2 Socioeconomic Factors
- Income, employment, education, and relationship status all appear uniformly distributed — a hallmark of synthetic generation.
- No natural skew toward any income bracket or education level.

### 3.3 Instagram Usage Patterns
- Average daily active time: **188 minutes** (median 186 min) — over 3 hours/day on average.
- Average sessions per day: **10.3**; average reels watched: **175.6/day**.
- Reels dominate time allocation across all activity types (Feed, Explore, Messages, Reels).
- Content type preferences are evenly spread across Photos, Videos, Stories, Reels, Mixed, and Other.

### 3.4 Usage by Generation
- Clear monotonic decline with age: Gen Z leads with **215 daily minutes** and **239 reels/day**; Boomer+ averages **155 minutes** and **133 reels/day**.
- Younger cohorts also post and story more actively.

---

## Section 4 — Correlation Analysis

### 4.1 Full Heatmap Observations
- The **Instagram Behaviour** cluster (21 features) is highly internally correlated — most pairs have r > 0.7.
- **Lifestyle & Health** features are almost entirely uncorrelated with each other and with the targets (r ≈ 0).
- This confirms that health variables carry little predictive signal in this dataset.

### 4.2 & 4.3 Feature–Target Correlations

**Self-Reported Happiness** — all top predictors are **negative**:
- `daily_active_minutes_instagram` r = −0.37, `likes_given_per_day` r = −0.37
- Every Instagram activity metric is negatively associated with happiness — no exceptions.

**Perceived Stress Score** — the strongest signal in the dataset, all **positive**:
- `daily_active_minutes_instagram` r = **+0.83** (strongest single predictor)
- `likes_given_per_day` +0.82, `stories_viewed_per_day` +0.81, `time_on_feed_per_day` +0.81
- Even passive consumption (ads viewed, reels watched) is strongly linked to stress.

**User Engagement Score** — all top predictors are **negative**:
- `stories_viewed_per_day` r = −0.56, `daily_active_minutes` r = −0.53
- Counter-intuitive direction suggests `user_engagement_score` encodes **quality/efficiency** of engagement, not raw volume.

### 4.4 Scatter Plots
- The Stress vs. Daily Usage scatter shows a clear positive band — visually confirming the r = 0.83 correlation.
- Age vs. Reels: visible downward slope — younger users watch significantly more reels.
- Exercise, sleep, and other lifestyle variables show essentially **flat / no trend** against happiness and stress.

---

## Section 5 — Lifestyle & Health Deep Dive

### 5.1 Well-being by Usage Category

| Usage Bucket | Avg Stress | Avg Happiness |
|---|---|---|
| Light (<30 min) | 4.6 | 7.8 |
| Moderate (30–60 min) | 7.2 | 6.7 |
| Heavy (1–2 h) | 10.3 | 6.0 |
| Very Heavy (2–3 h) | 15.9 | 5.6 |
| **Extreme (3 h+)** | **28.4** | **4.8** |

Both trends are **monotonic and steep** — stress increases 6× from Light to Extreme; happiness drops by 3 points.

### 5.2 Health Metric Distributions
- **BMI**, **blood pressure**, **daily steps**, and **sleep hours** all follow approximately uniform or normal distributions — expected from synthetic data.
- No pathological outliers are visible; all values fall within medically plausible ranges.

### 5.3 Diet Quality × Stress Level
- Every cell in the cross-tabulation is ≈ 25 % regardless of diet quality category.
- **Diet quality has zero relationship with stress** in this dataset — a clear artefact of synthetic generation where these two variables were drawn independently.

---

## Overall Takeaways

1. **Instagram usage is the dominant driver** of all three targets — the stronger the usage, the higher the stress, the lower the happiness, and paradoxically the lower the engagement score.
2. **Lifestyle and health variables** (sleep, exercise, diet, BMI) are essentially **noise** in this dataset — they carry no meaningful correlation with the targets.
3. **The dataset is fully synthetic** — uniform distributions across categorical features, independently generated columns, and demographic impossibilities (e.g. young teens with children) mean findings should not be generalised to real-world behaviour.
4. **High multicollinearity** within the Instagram Behaviour group means that modelling will require regularisation (Ridge/Lasso) or dimensionality reduction (PCA) to avoid overfitting.
5. **`user_engagement_score` behaves inversely** to raw usage volume, suggesting it was synthetically designed to represent engagement *quality* rather than *quantity*.

# 6. Feature Engineering

We create **23 new features** across 6 groups to enrich the dataset for downstream modelling:

| Group | # | Purpose |
|---|---|---|
| **Engagement Ratios** | 5 | Normalised interaction rates (likes/followers, DM reciprocity, ad CTR …) |
| **Usage Intensity** | 4 | Composite usage scores (session depth, passive vs active ratio …) |
| **Health Indices** | 4 | BMI score, cardiovascular score, activity score, overall wellness index |
| **Behavioural Segments** | 4 | Creator score, consumer score, social butterfly index, addiction risk score |
| **Time-based** | 3 | Account age, days since last login, account maturity bucket |
| **Lifestyle Interactions** | 3 | Work-life digital balance, offline/online social ratio, hobby–screen balance |

In [ ]:
import numpy as np

def engineer_features(dataframe):
    """Creates 23 new features across engagement, usage, health, behaviour, time, and lifestyle groups."""
    df_eng = dataframe.copy()

    # ------------------------------------------------------------------
    # 1. ENGAGEMENT RATIOS (5 features)
    # ------------------------------------------------------------------
    df_eng['likes_to_followers_ratio'] = (
        df_eng['likes_given_per_day'] / (df_eng['followers_count'] + 1)
    ).clip(0, 10)

    df_eng['comments_to_posts_ratio'] = (
        df_eng['comments_written_per_day'] / (df_eng['posts_created_per_week'] / 7 + 0.1)
    ).clip(0, 50)

    df_eng['follower_following_ratio'] = (
        df_eng['followers_count'] / (df_eng['following_count'] + 1)
    ).clip(0, 100)

    df_eng['dm_reciprocity'] = (
        df_eng['dms_received_per_week'] / (df_eng['dms_sent_per_week'] + 1)
    ).clip(0, 10)

    df_eng['ad_click_rate'] = (
        df_eng['ads_clicked_per_day'] / (df_eng['ads_viewed_per_day'] + 1)
    ).clip(0, 1)

    # ------------------------------------------------------------------
    # 2. USAGE INTENSITY SCORES (4 features)
    # ------------------------------------------------------------------
    df_eng['total_daily_interactions'] = (
        df_eng['likes_given_per_day'] +
        df_eng['comments_written_per_day'] +
        df_eng['stories_viewed_per_day'] +
        df_eng['reels_watched_per_day']
    )

    df_eng['usage_intensity_score'] = (
        df_eng['daily_active_minutes_instagram'] * df_eng['sessions_per_day']
    ) / 100

    df_eng['passive_active_ratio'] = (
        (df_eng['reels_watched_per_day'] + df_eng['stories_viewed_per_day']) /
        (df_eng['posts_created_per_week'] + df_eng['comments_written_per_day'] + 1)
    ).clip(0, 100)

    df_eng['session_depth'] = (
        df_eng['daily_active_minutes_instagram'] / (df_eng['sessions_per_day'] + 0.1)
    ).clip(0, 120)

    # ------------------------------------------------------------------
    # 3. HEALTH COMPOSITE INDICES (4 features)
    # ------------------------------------------------------------------
    df_eng['bmi_score'] = (100 - np.abs(df_eng['body_mass_index'] - 22) * 3).clip(0, 100)

    df_eng['cardio_score'] = (
        100
        - np.abs(df_eng['blood_pressure_systolic']  - 120) * 0.5
        - np.abs(df_eng['blood_pressure_diastolic'] - 80)  * 0.5
    ).clip(0, 100)

    df_eng['activity_score'] = (
        (df_eng['daily_steps_count'] / 100) +
        (df_eng['exercise_hours_per_week'] * 5)
    ).clip(0, 100)

    df_eng['wellness_index'] = (
        df_eng['bmi_score']      * 0.25 +
        df_eng['cardio_score']   * 0.25 +
        df_eng['activity_score'] * 0.25 +
        (df_eng['sleep_hours_per_night'] / 8 * 25)
    ).clip(0, 100)

    # ------------------------------------------------------------------
    # 4. BEHAVIOURAL SEGMENTS (4 features)
    # ------------------------------------------------------------------
    df_eng['creator_score'] = (
        df_eng['posts_created_per_week'] * 2 +
        df_eng['comments_written_per_day'] * 0.5
    ).clip(0, 50)

    df_eng['consumer_score'] = (
        df_eng['reels_watched_per_day']   * 0.3 +
        df_eng['stories_viewed_per_day']  * 0.3 +
        df_eng['time_on_explore_per_day'] * 0.4
    ).clip(0, 100)

    df_eng['social_butterfly_index'] = (
        df_eng['dms_sent_per_week'] +
        df_eng['dms_received_per_week'] +
        df_eng['comments_written_per_day'] * 7 +
        df_eng['social_events_per_month']  * 2
    ).clip(0, 200)

    df_eng['addiction_risk_score'] = (
        (df_eng['daily_active_minutes_instagram'] / 60) * 20 +
        df_eng['sessions_per_day'] * 3 +
        (df_eng['notification_response_rate'] * 20) +
        (40 - df_eng['perceived_stress_score']) * (-0.5)
    ).clip(0, 100)

    # ------------------------------------------------------------------
    # 5. TIME-BASED FEATURES (3 features)
    # ------------------------------------------------------------------
    current_year = 2025
    df_eng['account_age_years'] = current_year - df_eng['account_creation_year']

    df_eng['last_login_date'] = pd.to_datetime(df_eng['last_login_date'], errors='coerce')
    df_eng['days_since_last_login'] = (
        pd.to_datetime('2025-12-31') - df_eng['last_login_date']
    ).dt.days.fillna(0)

    df_eng['account_maturity'] = pd.cut(
        df_eng['account_age_years'],
        bins=[-1, 1, 3, 5, 10, 20],
        labels=['New (<1y)', 'Growing (1-3y)', 'Established (3-5y)', 'Veteran (5-10y)', 'Pioneer (10y+)']
    )

    # ------------------------------------------------------------------
    # 6. LIFESTYLE INTERACTION FEATURES (3 features)
    # ------------------------------------------------------------------
    df_eng['work_life_digital_balance'] = (
        df_eng['weekly_work_hours'] /
        (df_eng['daily_active_minutes_instagram'] * 7 / 60 + 1)
    ).clip(0, 50)

    df_eng['offline_online_social_ratio'] = (
        df_eng['social_events_per_month'] /
        (df_eng['sessions_per_day'] * 30 / 10 + 1)
    ).clip(0, 10)

    df_eng['hobby_screen_balance'] = (
        df_eng['hobbies_count'] * 10 /
        (df_eng['daily_active_minutes_instagram'] / 60 + 1)
    ).clip(0, 50)

    return df_eng


# Apply to full dataset and sample
print("Engineering features...")
df        = engineer_features(df)
df_sample = engineer_features(df_sample)

new_features = [
    'likes_to_followers_ratio', 'comments_to_posts_ratio', 'follower_following_ratio',
    'dm_reciprocity', 'ad_click_rate',
    'total_daily_interactions', 'usage_intensity_score', 'passive_active_ratio', 'session_depth',
    'bmi_score', 'cardio_score', 'activity_score', 'wellness_index',
    'creator_score', 'consumer_score', 'social_butterfly_index', 'addiction_risk_score',
    'account_age_years', 'days_since_last_login', 'account_maturity',
    'work_life_digital_balance', 'offline_online_social_ratio', 'hobby_screen_balance',
]

print(f"Created {len(new_features)} new engineered features!\n")
print(f"{'#':>3}  {'Feature':<35} {'Min':>8}  {'Mean':>8}  {'Max':>8}")
print("-" * 68)
for i, feat in enumerate(new_features, 1):
    if feat == 'account_maturity':
        print(f"{i:>3}  {feat:<35} {'(categorical)':>27}")
    else:
        s = df[feat]
        print(f"{i:>3}  {feat:<35} {s.min():>8.2f}  {s.mean():>8.2f}  {s.max():>8.2f}")

In [ ]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

scatter_eng = df_sample.sample(3000, random_state=42)

fig = make_subplots(
    rows=2, cols=3,
    subplot_titles=[
        'Addiction Risk Score', 'Wellness Index',
        'Creator vs Consumer Score', 'Passive / Active Ratio',
        'Social Butterfly Index', 'Usage Intensity Score',
    ]
)

# Addiction Risk
fig.add_trace(
    go.Histogram(x=df_sample['addiction_risk_score'],
                 marker_color=COLORS['secondary'], nbinsx=40, name='Addiction Risk'),
    row=1, col=1
)

# Wellness Index
fig.add_trace(
    go.Histogram(x=df_sample['wellness_index'],
                 marker_color=COLORS['success'], nbinsx=40, name='Wellness'),
    row=1, col=2
)

# Creator vs Consumer scatter
fig.add_trace(
    go.Scatter(
        x=scatter_eng['creator_score'], y=scatter_eng['consumer_score'],
        mode='markers',
        marker=dict(size=4, color=COLORS['primary'], opacity=0.5),
        name='Creator vs Consumer',
    ),
    row=1, col=3
)

# Passive / Active Ratio
fig.add_trace(
    go.Histogram(x=df_sample['passive_active_ratio'].clip(0, 30),
                 marker_color=COLORS['accent'], nbinsx=40, name='Passive/Active'),
    row=2, col=1
)

# Social Butterfly
fig.add_trace(
    go.Histogram(x=df_sample['social_butterfly_index'],
                 marker_color=COLORS['warning'], nbinsx=40, name='Social Butterfly'),
    row=2, col=2
)

# Usage Intensity
fig.add_trace(
    go.Histogram(x=df_sample['usage_intensity_score'].clip(0, 50),
                 marker_color=COLORS['info'], nbinsx=40, name='Usage Intensity'),
    row=2, col=3
)

fig.update_xaxes(title_text='Score', row=1, col=1)
fig.update_xaxes(title_text='Score', row=1, col=2)
fig.update_xaxes(title_text='Creator Score', row=1, col=3)
fig.update_yaxes(title_text='Consumer Score', row=1, col=3)
fig.update_xaxes(title_text='Ratio (clipped 0-30)', row=2, col=1)
fig.update_xaxes(title_text='Index', row=2, col=2)
fig.update_xaxes(title_text='Score (clipped 0-50)', row=2, col=3)

fig.update_layout(
    height=600, showlegend=False,
    title_text='<b>Distribution of Key Engineered Features</b>',
    title_x=0.5,
)
fig.show()

# 7. User Persona Clustering

We apply **K-Means** on the 50 % sample (~773 k rows) to discover natural user segments and assign them meaningful behavioural **personas**.

**Pipeline**
1. Select 15 representative features covering usage, content, social, health, and account dimensions.
2. Impute the rare missing values with column medians and standardise with `StandardScaler`.
3. Run an **elbow + silhouette sweep** on 20 000 rows to pick the statistically best *k*.
4. Fit final K-Means with **6 clusters** (chosen for persona richness) on the full 50 % sample.
5. Map each cluster to a named persona based on its profile.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import numpy as np
from plotly.subplots import make_subplots
import plotly.graph_objects as go

# -----------------------------------------------------------------------
# 7.1  Feature selection & scaling  (df_sample = 50 % of full dataset)
# -----------------------------------------------------------------------
clustering_features = [
    # Usage patterns
    'daily_active_minutes_instagram', 'sessions_per_day', 'average_session_length_minutes',
    # Content behaviour
    'creator_score', 'consumer_score', 'passive_active_ratio',
    # Social behaviour
    'social_butterfly_index', 'dm_reciprocity',
    # Health / Wellness
    'wellness_index', 'perceived_stress_score', 'self_reported_happiness',
    # Account
    'followers_count', 'follower_following_ratio',
    # Engineered
    'addiction_risk_score', 'usage_intensity_score',
]

X_cluster = df_sample[clustering_features].copy().fillna(df_sample[clustering_features].median())

scaler   = StandardScaler()
X_scaled = scaler.fit_transform(X_cluster)

print(f"Prepared {X_scaled.shape[0]:,} samples × {X_scaled.shape[1]} features for clustering")

# -----------------------------------------------------------------------
# 7.2  Elbow + silhouette sweep on 20 000 rows
# -----------------------------------------------------------------------
X_opt = X_scaled[:20_000]

k_range          = range(2, 11)
inertias         = []
silhouette_scores = []

print("\nFinding optimal k …")
for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10, max_iter=300)
    km.fit(X_opt)
    inertias.append(km.inertia_)
    sil = silhouette_score(X_opt, km.labels_)
    silhouette_scores.append(sil)
    print(f"   k={k}: silhouette = {sil:.4f}")

best_k = list(k_range)[int(np.argmax(silhouette_scores))]
print(f"\nOptimal k (highest silhouette): {best_k}  |  score = {max(silhouette_scores):.4f}")

# -----------------------------------------------------------------------
# Plot
# -----------------------------------------------------------------------
fig = make_subplots(rows=1, cols=2,
                    subplot_titles=['Elbow Method (Inertia)', 'Silhouette Score'])

fig.add_trace(go.Scatter(x=list(k_range), y=inertias, mode='lines+markers',
                         marker=dict(color=COLORS['primary'], size=10),
                         line=dict(color=COLORS['primary'], width=2),
                         name='Inertia'),
              row=1, col=1)

fig.add_trace(go.Scatter(x=list(k_range), y=silhouette_scores, mode='lines+markers',
                         marker=dict(color=COLORS['secondary'], size=10),
                         line=dict(color=COLORS['secondary'], width=2),
                         name='Silhouette'),
              row=1, col=2)

# Mark best k on silhouette panel
fig.add_trace(go.Scatter(x=[best_k], y=[max(silhouette_scores)], mode='markers',
                         marker=dict(color=COLORS['success'], size=14, symbol='star'),
                         name=f'Best k={best_k}'),
              row=1, col=2)

fig.update_xaxes(title_text='Number of Clusters (k)')
fig.update_yaxes(title_text='Inertia',          row=1, col=1)
fig.update_yaxes(title_text='Silhouette Score', row=1, col=2)
fig.update_layout(height=420, showlegend=False,
                  title_text='<b>Optimal Cluster Selection</b>', title_x=0.5)
fig.show()

In [ ]:
import plotly.express as px

# -----------------------------------------------------------------------
# 7.3  Final K-Means with 6 clusters on full 50 % sample
# -----------------------------------------------------------------------
N_CLUSTERS = 6  # chosen for meaningful persona variety

kmeans_final = KMeans(n_clusters=N_CLUSTERS, random_state=42, n_init=10, max_iter=300)
df_sample['cluster'] = kmeans_final.fit_predict(X_scaled)

# Cluster profiles
cluster_profiles = df_sample.groupby('cluster', observed=True)[clustering_features].mean()
print("Cluster Profiles (mean values):")
print(cluster_profiles.round(2).T.to_string())

# -----------------------------------------------------------------------
# 7.4  Persona definitions
#      Colours mapped to our palette (no 'danger' key in COLORS)
# -----------------------------------------------------------------------
persona_definitions = {
    0: {'name': 'The Healthy Casual',       'emoji': '🌿', 'color': COLORS['success']},
    1: {'name': 'The Doom Scroller',         'emoji': '📱', 'color': COLORS['secondary']},
    2: {'name': 'The Aspiring Influencer',   'emoji': '⭐', 'color': COLORS['warning']},
    3: {'name': 'The Social Butterfly',      'emoji': '🦋', 'color': COLORS['primary']},
    4: {'name': 'The Lurker',                'emoji': '👀', 'color': COLORS['info']},
    5: {'name': 'The Power User',            'emoji': '🚀', 'color': COLORS['accent']},
}

persona_descriptions = {
    'The Healthy Casual':      'Balanced users with moderate usage, good wellness, healthy offline lives.',
    'The Doom Scroller':       'High usage, high stress, passive consumption — hours of reels & stories.',
    'The Aspiring Influencer': 'Active creators with high posting frequency and engagement focus.',
    'The Social Butterfly':    'Connection-focused users who prioritise DMs, comments & social events.',
    'The Lurker':              'Low engagement, minimal posting, regular silent consumption.',
    'The Power User':          'High followers, high engagement — potential micro-influencers.',
}

df_sample['persona'] = df_sample['cluster'].map(lambda x: persona_definitions[x]['name'])

# Distribution summary
print("\nPersona Distribution:")
persona_counts = df_sample['persona'].value_counts()
for persona, count in persona_counts.items():
    pct   = count / len(df_sample) * 100
    emoji = next(v['emoji'] for v in persona_definitions.values() if v['name'] == persona)
    print(f"   {emoji} {persona:<30} {count:>8,}  ({pct:.1f}%)")

# -----------------------------------------------------------------------
# 7.5  Visualisation: donut + radar profile
# -----------------------------------------------------------------------
ordered_clusters = list(range(N_CLUSTERS))
pie_labels  = [f"{persona_definitions[i]['emoji']} {persona_definitions[i]['name']}" for i in ordered_clusters]
pie_values  = df_sample['cluster'].value_counts().sort_index().values
pie_colors  = [persona_definitions[i]['color'] for i in ordered_clusters]

fig_pie = go.Figure(go.Pie(
    labels=pie_labels,
    values=pie_values,
    marker_colors=pie_colors,
    hole=0.45,
    textinfo='label+percent',
    hovertemplate='<b>%{label}</b><br>Users: %{value:,}<br>Share: %{percent}<extra></extra>',
))
fig_pie.update_layout(
    title_text='<b>User Persona Distribution</b>', title_x=0.5,
    height=520, showlegend=True,
    legend=dict(orientation='v', x=1.02, y=0.5),
)
fig_pie.show()

# -----------------------------------------------------------------------
# Radar chart — normalised cluster profiles
# -----------------------------------------------------------------------
radar_features = [
    'daily_active_minutes_instagram', 'creator_score', 'consumer_score',
    'wellness_index', 'social_butterfly_index', 'addiction_risk_score',
]
radar_labels = ['Daily Minutes', 'Creator', 'Consumer', 'Wellness', 'Social', 'Addiction Risk']

# Min-max normalise across clusters for radar readability
prof_radar = cluster_profiles[radar_features].copy()
prof_radar = (prof_radar - prof_radar.min()) / (prof_radar.max() - prof_radar.min() + 1e-9)

fig_radar = go.Figure()
for i in ordered_clusters:
    vals = prof_radar.loc[i, radar_features].tolist()
    vals += [vals[0]]   # close the polygon
    fig_radar.add_trace(go.Scatterpolar(
        r=vals,
        theta=radar_labels + [radar_labels[0]],
        fill='toself',
        name=f"{persona_definitions[i]['emoji']} {persona_definitions[i]['name']}",
        line=dict(color=persona_definitions[i]['color']),
        fillcolor=persona_definitions[i]['color'],
        opacity=0.25,
    ))

fig_radar.update_layout(
    polar=dict(radialaxis=dict(visible=True, range=[0, 1])),
    title_text='<b>Persona Radar Profiles (normalised)</b>', title_x=0.5,
    height=550, showlegend=True,
)
fig_radar.show()

In [ ]:
from sklearn.decomposition import PCA
import plotly.express as px

# -----------------------------------------------------------------------
# 7.6  PCA — 2-D visualisation of clusters
# -----------------------------------------------------------------------
pca   = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

viz_df = pd.DataFrame({
    'PC1':     X_pca[:, 0],
    'PC2':     X_pca[:, 1],
    'Cluster': df_sample['cluster'].values,
    'Persona': df_sample['persona'].values,
})

color_map = {persona_definitions[i]['name']: persona_definitions[i]['color']
             for i in range(N_CLUSTERS)}

fig = px.scatter(
    viz_df.sample(10_000, random_state=42),
    x='PC1', y='PC2',
    color='Persona',
    color_discrete_map=color_map,
    title='<b>User Personas Visualised via PCA</b>',
    opacity=0.6,
)
fig.update_layout(
    height=600,
    xaxis_title=f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)",
    yaxis_title=f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)",
    title_x=0.5,
)
fig.show()

print(f"PCA explained variance: {sum(pca.explained_variance_ratio_)*100:.1f}% with 2 components")

# -----------------------------------------------------------------------
# 7.7  Radar chart — persona comparison (stress instead of addiction risk)
# -----------------------------------------------------------------------
radar_metrics = [
    'daily_active_minutes_instagram', 'creator_score', 'consumer_score',
    'wellness_index', 'perceived_stress_score', 'social_butterfly_index',
]
radar_labels_v2 = ['Daily Minutes', 'Creator', 'Consumer', 'Wellness', 'Stress', 'Social']

radar_data       = df_sample.groupby('cluster', observed=True)[radar_metrics].mean()
radar_normalised = (radar_data - radar_data.min()) / (radar_data.max() - radar_data.min() + 1e-9)

fig_r2 = go.Figure()
for i in range(N_CLUSTERS):
    vals = radar_normalised.loc[i].tolist()
    vals += [vals[0]]
    fig_r2.add_trace(go.Scatterpolar(
        r=vals,
        theta=radar_labels_v2 + [radar_labels_v2[0]],
        fill='toself',
        name=f"{persona_definitions[i]['emoji']} {persona_definitions[i]['name']}",
        line_color=persona_definitions[i]['color'],
        opacity=0.7,
    ))

fig_r2.update_layout(
    polar=dict(radialaxis=dict(visible=True, range=[0, 1])),
    showlegend=True,
    title='<b>Persona Comparison Radar Chart</b>',
    title_x=0.5,
    height=600,
)
fig_r2.show()

# -----------------------------------------------------------------------
# 7.8  Detailed persona profiles
# -----------------------------------------------------------------------
print("=" * 80)
print("DETAILED PERSONA PROFILES")
print("=" * 80)

for cluster_id in range(N_CLUSTERS):
    cluster_data = df_sample[df_sample['cluster'] == cluster_id]
    persona      = persona_definitions[cluster_id]
    description  = persona_descriptions[persona['name']]

    print(f"\n{persona['emoji']}  {persona['name'].upper()}")
    print("-" * 50)
    print(f"Description : {description}")
    print(f"Population  : {len(cluster_data):,} users  "
          f"({len(cluster_data)/len(df_sample)*100:.1f}%)")
    print(f"\nKey Metrics:")
    print(f"  • Daily Instagram Minutes : {cluster_data['daily_active_minutes_instagram'].mean():.0f}")
    print(f"  • Sessions per Day        : {cluster_data['sessions_per_day'].mean():.1f}")
    print(f"  • Stress Score            : {cluster_data['perceived_stress_score'].mean():.1f}")
    print(f"  • Happiness Score         : {cluster_data['self_reported_happiness'].mean():.1f}")
    print(f"  • Wellness Index          : {cluster_data['wellness_index'].mean():.1f}")
    print(f"  • Addiction Risk          : {cluster_data['addiction_risk_score'].mean():.1f}")
    print(f"\nDemographics:")
    print(f"  • Avg Age      : {cluster_data['age'].mean():.0f} years")
    print(f"  • Top Gender   : {cluster_data['gender'].mode().iloc[0]}")
    print(f"  • Top Country  : {cluster_data['country'].mode().iloc[0]}")

# Section 7 — Key Findings: User Persona Clustering

## Methodology
- **Algorithm:** K-Means on the 50 % sample (773 948 rows × 15 features)
- **Features:** usage intensity, content behaviour (creator/consumer scores), social behaviour (DM reciprocity, social butterfly index), health (wellness index, stress, happiness), account metrics (followers, follower ratio), and engineered scores (addiction risk, usage intensity)
- **Preprocessing:** median imputation + `StandardScaler` standardisation
- **Cluster selection:** elbow + silhouette sweep (k = 2–10 on 20 000 rows) → best silhouette = **0.277 at k = 2**; **k = 6** chosen for persona richness
- **Dimensionality reduction:** PCA (2 components) captures **53.2 %** of total variance

---

## Persona Profiles

| Persona | Share | Defining traits |
|---|---|---|
| 📱 **The Doom Scroller** | 28.0 % | High daily minutes (281), high stress (29.8), maxed-out addiction risk |
| ⭐ **The Aspiring Influencer** | 25.4 % | Moderate usage (170 min), strong creator score, lower stress (17.7) |
| 👀 **The Lurker** | 19.4 % | Low usage (91 min, 4.2 sessions), highest happiness (5.8), low stress (9.9) |
| 🦋 **The Social Butterfly** | 13.2 % | Most sessions/day (25.7), maxed addiction risk (100), lowest happiness (4.1) |
| 🚀 **The Power User** | 12.5 % | Highest followers (avg 21 k), minimal usage (22 min), lowest stress (5.5), highest happiness (7.6) |
| 🌿 **The Healthy Casual** | 1.4 % | High followers (21 k), moderate usage, balanced well-being; rarest cluster |

---

## Key Observations

1. **Usage volume ≠ happiness.** The Lurker (least active) is the happiest; the Social Butterfly (most sessions) is the least happy.
2. **The Power User is an outlier.** Far higher followers than every other cluster — likely professional/business accounts that use Instagram purposefully and briefly.
3. **Addiction risk tracks stress tightly.** The two clusters with maxed-out addiction risk (Doom Scroller, Social Butterfly) also show the highest stress scores (29.8 and 31.7).
4. **The Aspiring Influencer is the "healthiest" active user** — moderate usage, strong creative output, and below-average stress.
5. **PCA separability is moderate** (53.2 % variance in 2D) — clusters show broad spatial separation driven primarily by usage intensity and follower count axes, with significant overlap in the middle ground.
6. **Synthetic data caveat:** wellness index is nearly identical across all clusters (~89.9), confirming that health features were generated independently of behaviour — a known limitation of the dataset.

# 8. Predictive Modeling

We build and compare models for **four prediction tasks** on the 50 % sample (~773 k rows), split 80 / 20 into train and test sets.

| # | Task | Type | Target |
|---|---|---|---|
| 1 | Self-Reported Happiness | Regression | `self_reported_happiness` |
| 2 | Perceived Stress Score | Regression | `perceived_stress_score` |
| 3 | User Engagement Score | Regression | `user_engagement_score` |
| 4 | Social Media Addiction Risk | Binary Classification | `addiction_risk_score ≥ 50` |

**Pipeline**
1. **Feature preparation** — label-encode all categorical columns; assemble final feature matrix.
2. **Train / test split** — stratified 80 / 20 on `df_sample`.
3. **Scaling** — `StandardScaler` fit on train, applied to both splits.
4. **Model comparison** — four algorithms per task evaluated on the held-out test set (R² / RMSE / MAE for regression; Accuracy / F1 / AUC-ROC for classification).
5. **Best model selection** — winner chosen per task based on primary metric.

## 8.1 Feature Preparation and Split
We prepare a modeling dataset from the 50% sample, encode categorical variables, create the binary addiction-risk target, and then perform an 80/20 train-test split before scaling the features.

In [ ]:
# Import the preprocessing, modeling, and evaluation tools used in this section.
from sklearn.ensemble import (
    HistGradientBoostingClassifier,
    HistGradientBoostingRegressor,
    RandomForestClassifier,
    RandomForestRegressor,
)
from sklearn.linear_model import LinearRegression, LogisticRegression, Ridge
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler

# Reuse the 50% sample if it already exists; otherwise create it once from the full dataset.
modeling_df = df_sample.copy() if 'df_sample' in globals() else df.sample(frac=0.5, random_state=42).copy()

# Exclude identifiers, engineered segmentation columns, and prediction targets from the feature pool.
exclude_cols = [
    'user_id', 'app_name', 'self_reported_happiness', 'perceived_stress_score',
    'user_engagement_score', 'addiction_risk_score', 'last_login_date', 'cluster',
    'persona', 'generation', 'usage_category', 'stress_category', 'account_maturity'
]

# Keep the remaining numeric columns as the starting point for the feature set.
feature_cols = [
    col for col in modeling_df.columns
    if col not in exclude_cols and modeling_df[col].dtype in ['int64', 'float64']
]

# Encode the categorical variables so that every predictor is numeric before training.
categorical_cols = [
    'gender', 'country', 'urban_rural', 'income_level', 'employment_status',
    'education_level', 'relationship_status', 'has_children', 'diet_quality',
    'smoking', 'alcohol_frequency', 'content_type_preference',
    'preferred_content_theme', 'privacy_setting_level', 'two_factor_auth_enabled',
    'biometric_login_used', 'subscription_status', 'uses_premium_features'
]

label_encoders = {}
for col in categorical_cols:
    if col in modeling_df.columns:
        encoder = LabelEncoder()
        encoded_col = f'{col}_encoded'
        modeling_df[encoded_col] = encoder.fit_transform(modeling_df[col].astype(str))
        label_encoders[col] = encoder
        feature_cols.append(encoded_col)

# Finalize the feature matrix and replace any remaining invalid values.
feature_cols = sorted(set(feature_cols))
X = modeling_df[feature_cols].replace([float('inf'), float('-inf')], 0).fillna(0)

# Define the three regression targets plus the binary addiction-risk classification target.
y_targets = {
    'Self-Reported Happiness': modeling_df['self_reported_happiness'],
    'Perceived Stress Score': modeling_df['perceived_stress_score'],
    'User Engagement Score': modeling_df['user_engagement_score'],
    'Social Media Addiction Risk': (modeling_df['addiction_risk_score'] >= 50).astype(int),
}

# Split the 50% sample into 80% train and 20% test, stratifying on addiction risk.
train_idx, test_idx = train_test_split(
    modeling_df.index,
    test_size=0.2,
    random_state=42,
    stratify=y_targets['Social Media Addiction Risk'],
)

# Scale the features using statistics learned from the training data only.
scaler_modeling = StandardScaler()
X_train = scaler_modeling.fit_transform(X.loc[train_idx])
X_test = scaler_modeling.transform(X.loc[test_idx])

y_train = {name: values.loc[train_idx] for name, values in y_targets.items()}
y_test = {name: values.loc[test_idx] for name, values in y_targets.items()}

print(f'Prepared {X.shape[1]} features for modeling.')
print(f'Modeling sample size: {X.shape[0]:,} rows.')
print(f'Training rows: {len(train_idx):,} | Test rows: {len(test_idx):,}')
print(
    'Classification positive rate (train/test): '
    f"{y_train['Social Media Addiction Risk'].mean():.3f} / {y_test['Social Media Addiction Risk'].mean():.3f}"
)

# Define the candidate regression models to compare on the held-out test set.
regression_models = {
    'Linear Regression': LinearRegression(),
    'Ridge Regression': Ridge(alpha=1.0),
    'Random Forest Regressor': RandomForestRegressor(
        n_estimators=120,
        max_depth=12,
        min_samples_leaf=4,
        random_state=42,
        n_jobs=-1,
    ),
    'HistGradientBoosting Regressor': HistGradientBoostingRegressor(
        max_depth=10,
        learning_rate=0.08,
        random_state=42,
    ),
}

# Define the candidate classification models for addiction-risk prediction.
classification_models = {
    'Logistic Regression': LogisticRegression(max_iter=1000),
    'Random Forest Classifier': RandomForestClassifier(
        n_estimators=120,
        max_depth=12,
        min_samples_leaf=4,
        random_state=42,
        n_jobs=-1,
    ),
    'HistGradientBoosting Classifier': HistGradientBoostingClassifier(
        max_depth=10,
        learning_rate=0.08,
        random_state=42,
    ),
}

## 8.2 Model Performance Visualizations

The plots below use the best-performing model for each target from the comparison step. For the three regression targets, we compare predicted values against the true values on the held-out test set. For the addiction-risk classifier, we inspect the confusion matrix, ROC curve, and predicted probability separation.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.base import clone
from sklearn.metrics import confusion_matrix, roc_curve

# Refit the best-performing model for each target so the diagnostics below are reproducible.
best_model_lookup = {
    'Self-Reported Happiness': 'HistGradientBoosting Regressor',
    'Perceived Stress Score': 'HistGradientBoosting Regressor',
    'User Engagement Score': 'HistGradientBoosting Regressor',
    'Social Media Addiction Risk': 'HistGradientBoosting Classifier',
}

fitted_best_models = {}
regression_predictions = {}
classification_predictions = {}
classification_probabilities = {}

for target_name in ['Self-Reported Happiness', 'Perceived Stress Score', 'User Engagement Score']:
    model_name = best_model_lookup[target_name]
    model = clone(regression_models[model_name])
    model.fit(X_train, y_train[target_name])
    fitted_best_models[target_name] = model
    regression_predictions[target_name] = model.predict(X_test)

classification_target = 'Social Media Addiction Risk'
classification_model = clone(classification_models[best_model_lookup[classification_target]])
classification_model.fit(X_train, y_train[classification_target])
fitted_best_models[classification_target] = classification_model
classification_predictions[classification_target] = classification_model.predict(X_test)
classification_probabilities[classification_target] = classification_model.predict_proba(X_test)[:, 1]

# Use a smaller random slice for scatter plots so they stay readable.
rng = np.random.default_rng(42)
scatter_size = min(5000, X_test.shape[0])
scatter_idx = rng.choice(X_test.shape[0], size=scatter_size, replace=False)

fig, axes = plt.subplots(2, 3, figsize=(18, 11))
fig.suptitle('Best Model Diagnostics on the Test Set', fontsize=16, fontweight='bold', y=1.02)

regression_targets = [
    'Self-Reported Happiness',
    'Perceived Stress Score',
    'User Engagement Score',
]

for axis, target_name in zip(axes[0], regression_targets):
    actual = y_test[target_name].iloc[scatter_idx].to_numpy()
    predicted = regression_predictions[target_name][scatter_idx]
    metrics_row = best_regression_models.loc[best_regression_models['Target'] == target_name].iloc[0]
    min_val = min(actual.min(), predicted.min())
    max_val = max(actual.max(), predicted.max())

    axis.scatter(actual, predicted, s=10, alpha=0.25, color=COLORS['primary'])
    axis.plot([min_val, max_val], [min_val, max_val], linestyle='--', color='black', linewidth=1)
    axis.set_title(f"{target_name}\nR2={metrics_row['R2']:.3f} | RMSE={metrics_row['RMSE']:.3f}")
    axis.set_xlabel('Actual')
    axis.set_ylabel('Predicted')

conf_matrix = confusion_matrix(
    y_test[classification_target],
    classification_predictions[classification_target],
    normalize='true',
)

sns.heatmap(
    conf_matrix,
    annot=True,
    fmt='.2f',
    cmap='Blues',
    cbar=False,
    ax=axes[1, 0],
)
axes[1, 0].set_title('Addiction Risk Classification\nNormalized Confusion Matrix')
axes[1, 0].set_xlabel('Predicted label')
axes[1, 0].set_ylabel('True label')
axes[1, 0].set_xticklabels(['Low Risk', 'High Risk'])
axes[1, 0].set_yticklabels(['Low Risk', 'High Risk'], rotation=0)

fpr, tpr, _ = roc_curve(y_test[classification_target], classification_probabilities[classification_target])
roc_auc = best_classification_model.iloc[0]['ROC AUC']
axes[1, 1].plot(fpr, tpr, color=COLORS['secondary'], linewidth=2, label=f'AUC = {roc_auc:.4f}')
axes[1, 1].plot([0, 1], [0, 1], linestyle='--', color='gray', linewidth=1)
axes[1, 1].set_title('Addiction Risk Classification\nROC Curve')
axes[1, 1].set_xlabel('False Positive Rate')
axes[1, 1].set_ylabel('True Positive Rate')
axes[1, 1].legend(loc='lower right')

probabilities = classification_probabilities[classification_target]
actual_classes = y_test[classification_target].to_numpy()
axes[1, 2].hist(probabilities[actual_classes == 0], bins=30, alpha=0.6, label='Actual Low Risk', color=COLORS['info'])
axes[1, 2].hist(probabilities[actual_classes == 1], bins=30, alpha=0.6, label='Actual High Risk', color=COLORS['accent'])
axes[1, 2].set_title('Addiction Risk Classification\nPredicted Probability Separation')
axes[1, 2].set_xlabel('Predicted probability of high risk')
axes[1, 2].set_ylabel('Count')
axes[1, 2].legend()

plt.tight_layout()
plt.show()

## 8.3 Most Influential Features Per Target

Because the strongest models are gradient-boosting models, we use permutation importance on a held-out test subset to estimate which features matter most for each target. Higher values indicate a larger drop in model performance when that feature is shuffled.

In [ ]:
import pandas as pd
from sklearn.inspection import permutation_importance

if 'fitted_best_models' not in globals():
    raise RuntimeError('Run the previous visualization cell first so the best models are fitted.')

feature_names = X.columns.tolist()
importance_sample_size = min(4000, X_test.shape[0])
rng = np.random.default_rng(42)
importance_idx = rng.choice(X_test.shape[0], size=importance_sample_size, replace=False)
X_importance = X_test[importance_idx]

importance_specs = {
    'Self-Reported Happiness': ('r2', y_test['Self-Reported Happiness'].iloc[importance_idx]),
    'Perceived Stress Score': ('r2', y_test['Perceived Stress Score'].iloc[importance_idx]),
    'User Engagement Score': ('r2', y_test['User Engagement Score'].iloc[importance_idx]),
    'Social Media Addiction Risk': ('roc_auc', y_test['Social Media Addiction Risk'].iloc[importance_idx]),
}

feature_importance_summary = []
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('Permutation Importance of the Best Model for Each Target', fontsize=16, fontweight='bold', y=1.02)

for axis, (target_name, (scoring_name, y_subset)) in zip(axes.flatten(), importance_specs.items()):
    importance_result = permutation_importance(
        fitted_best_models[target_name],
        X_importance,
        y_subset,
        n_repeats=5,
        random_state=42,
        scoring=scoring_name,
        n_jobs=-1,
    )

    importance_df = pd.DataFrame({
        'Feature': feature_names,
        'Importance': importance_result.importances_mean,
    }).sort_values('Importance', ascending=False).head(12)

    feature_importance_summary.append(
        importance_df.assign(Target=target_name)[['Target', 'Feature', 'Importance']]
    )

    plot_df = importance_df.sort_values('Importance', ascending=True)
    axis.barh(plot_df['Feature'], plot_df['Importance'], color=COLORS['secondary'])
    axis.set_title(target_name)
    axis.set_xlabel(f'Permutation importance ({scoring_name})')
    axis.set_ylabel('Feature')

plt.tight_layout()
plt.show()

feature_importance_summary = pd.concat(feature_importance_summary, ignore_index=True)
display(feature_importance_summary.groupby('Target', group_keys=False).head(8))

# 9. Advanced Analysis: Causal Inference

## 9.1 Does Instagram Usage Cause Lower Happiness?

This section moves beyond prediction and asks a harder question: is heavier Instagram usage plausibly linked to *lower happiness after accounting for confounding factors*, or are less happy users simply more likely to spend more time on the platform?

Because this dataset is synthetic and observational, we cannot claim true causality. Instead, we run a set of **causal-style diagnostics** that are coherent with the rest of the project:

1. **Confounder audit** — identify variables related to both Instagram usage and happiness.
2. **Adjusted regression** — compare the raw usage-happiness relationship with the relationship after controlling for age, stress, sleep, exercise, offline social activity, work hours, and socioeconomic context.
3. **Stratified analysis** — check whether the usage-happiness pattern remains inside comparable stress groups.
4. **Quasi-experimental account-maturity view** — test whether the relationship changes across newer and older accounts.

The goal here is not to prove causation, but to assess whether the negative usage-happiness relationship survives basic attempts to control for alternative explanations.

In [ ]:
from sklearn.linear_model import LinearRegression

# Build a focused causal-analysis table from the 50% sample used throughout the notebook.
causal_df = df_sample.copy()

usage_col = 'daily_active_minutes_instagram'
outcome_col = 'self_reported_happiness'

# Encode the socioeconomic variables locally so this section does not depend on the modeling cell mutating df_sample.
socioeconomic_categoricals = [
    'income_level',
    'employment_status',
    'education_level',
    'relationship_status',
]
for col in socioeconomic_categoricals:
    encoded_col = f'{col}_encoded'
    if encoded_col not in causal_df.columns and col in causal_df.columns:
        encoder = LabelEncoder()
        causal_df[encoded_col] = encoder.fit_transform(causal_df[col].astype(str))

confounders = [
    'age',
    'perceived_stress_score',
    'sleep_hours_per_night',
    'exercise_hours_per_week',
    'social_events_per_month',
    'weekly_work_hours',
    'income_level_encoded',
    'employment_status_encoded',
    'education_level_encoded',
    'relationship_status_encoded',
    'hobbies_count',
    'wellness_index',
]

analysis_cols = [usage_col, outcome_col] + confounders + ['account_maturity']
causal_df = causal_df[analysis_cols].replace([np.inf, -np.inf], np.nan).dropna().copy()

# ---------------------------------------------------------------------
# 1. Confounder audit: which variables are linked to both usage and happiness?
# ---------------------------------------------------------------------
confounder_audit = pd.DataFrame({
    'Confounder': confounders,
    'Corr with Usage': [causal_df[col].corr(causal_df[usage_col]) for col in confounders],
    'Corr with Happiness': [causal_df[col].corr(causal_df[outcome_col]) for col in confounders],
})
confounder_audit['Combined Signal'] = (
    confounder_audit['Corr with Usage'].abs() * confounder_audit['Corr with Happiness'].abs()
)
confounder_audit = confounder_audit.sort_values('Combined Signal', ascending=False)

# ---------------------------------------------------------------------
# 2. Naive vs adjusted relationship: does usage still matter after controls?
# ---------------------------------------------------------------------
naive_model = LinearRegression()
naive_model.fit(causal_df[[usage_col]], causal_df[outcome_col])
naive_slope = float(naive_model.coef_[0])
naive_r2 = float(naive_model.score(causal_df[[usage_col]], causal_df[outcome_col]))

adjusted_model = LinearRegression()
adjusted_model.fit(causal_df[[usage_col] + confounders], causal_df[outcome_col])
adjusted_slope = float(adjusted_model.coef_[0])
adjusted_r2 = float(adjusted_model.score(causal_df[[usage_col] + confounders], causal_df[outcome_col]))

# Residualize both treatment and outcome against confounders to estimate the partial association.
usage_residual_model = LinearRegression().fit(causal_df[confounders], causal_df[usage_col])
outcome_residual_model = LinearRegression().fit(causal_df[confounders], causal_df[outcome_col])
causal_df['usage_residual'] = causal_df[usage_col] - usage_residual_model.predict(causal_df[confounders])
causal_df['happiness_residual'] = causal_df[outcome_col] - outcome_residual_model.predict(causal_df[confounders])
partial_corr = causal_df['usage_residual'].corr(causal_df['happiness_residual'])

# ---------------------------------------------------------------------
# 3. Stratified analysis: compare usage quartiles within stress tertiles.
# ---------------------------------------------------------------------
causal_df['stress_group'] = pd.qcut(
    causal_df['perceived_stress_score'],
    q=3,
    labels=['Lower Stress', 'Middle Stress', 'Higher Stress'],
)
causal_df['usage_group'] = pd.qcut(
    causal_df[usage_col],
    q=4,
    labels=['Light', 'Moderate', 'Heavy', 'Very Heavy'],
)

stratified_summary = (
    causal_df.groupby(['stress_group', 'usage_group'], observed=True)
    .agg(
        mean_happiness=(outcome_col, 'mean'),
        mean_usage=(usage_col, 'mean'),
        sample_size=(outcome_col, 'size'),
    )
    .reset_index()
)

within_stress_corr = (
    causal_df.groupby('stress_group', observed=True)
    .apply(lambda frame: frame[usage_col].corr(frame[outcome_col]), include_groups=False)
    .rename('Usage-Happiness Correlation')
    .reset_index()
)

# ---------------------------------------------------------------------
# 4. Quasi-experimental view: compare the pattern across account maturity buckets.
# ---------------------------------------------------------------------
account_maturity_summary = (
    causal_df.groupby('account_maturity', observed=True)
    .apply(
        lambda frame: pd.Series({
            'Usage-Happiness Correlation': frame[usage_col].corr(frame[outcome_col]),
            'Average Usage Minutes': frame[usage_col].mean(),
            'Average Happiness': frame[outcome_col].mean(),
            'Sample Size': len(frame),
        }),
        include_groups=False,
    )
    .reset_index()
)

# ---------------------------------------------------------------------
# Visual summary of the causal-style checks.
# ---------------------------------------------------------------------
fig, axes = plt.subplots(2, 2, figsize=(18, 12))
fig.suptitle('Causal-Style Diagnostics: Instagram Usage and Happiness', fontsize=16, fontweight='bold', y=1.02)

plot_audit = confounder_audit.head(8).sort_values('Combined Signal')
axes[0, 0].barh(plot_audit['Confounder'], plot_audit['Corr with Usage'], color=COLORS['secondary'], alpha=0.8, label='Usage')
axes[0, 0].barh(plot_audit['Confounder'], plot_audit['Corr with Happiness'], color=COLORS['success'], alpha=0.6, label='Happiness')
axes[0, 0].set_title('Top Potential Confounders')
axes[0, 0].set_xlabel('Pearson correlation')
axes[0, 0].legend()

residual_sample = causal_df.sample(min(5000, len(causal_df)), random_state=42)
axes[0, 1].scatter(
    residual_sample['usage_residual'],
    residual_sample['happiness_residual'],
    s=10,
    alpha=0.25,
    color=COLORS['primary'],
)
trend = np.polyfit(residual_sample['usage_residual'], residual_sample['happiness_residual'], 1)
trend_x = np.linspace(residual_sample['usage_residual'].min(), residual_sample['usage_residual'].max(), 100)
axes[0, 1].plot(trend_x, trend[0] * trend_x + trend[1], color='black', linestyle='--', linewidth=1.5)
axes[0, 1].set_title(f'Partial Relationship After Controls\nResidual correlation = {partial_corr:+.3f}')
axes[0, 1].set_xlabel('Usage residual after controls')
axes[0, 1].set_ylabel('Happiness residual after controls')

sns.lineplot(
    data=stratified_summary,
    x='usage_group',
    y='mean_happiness',
    hue='stress_group',
    marker='o',
    palette=[COLORS['info'], COLORS['warning'], COLORS['secondary']],
    ax=axes[1, 0],
)
axes[1, 0].set_title('Mean Happiness by Usage Group Within Stress Tertiles')
axes[1, 0].set_xlabel('Instagram usage group')
axes[1, 0].set_ylabel('Average happiness')
axes[1, 0].legend(title='Stress group')

account_plot = account_maturity_summary.dropna().copy()
sns.scatterplot(
    data=account_plot,
    x='Average Usage Minutes',
    y='Average Happiness',
    size='Sample Size',
    hue='Usage-Happiness Correlation',
    palette='coolwarm_r',
    sizes=(80, 400),
    ax=axes[1, 1],
)
for _, row in account_plot.iterrows():
    axes[1, 1].text(row['Average Usage Minutes'] + 1, row['Average Happiness'] + 0.01, row['account_maturity'], fontsize=8)
axes[1, 1].set_title('Quasi-Experimental View by Account Maturity')
axes[1, 1].set_xlabel('Average daily Instagram minutes')
axes[1, 1].set_ylabel('Average happiness')

plt.tight_layout()
plt.show()

print('=' * 88)
print('CAUSAL INFERENCE SUMMARY: Instagram Usage and Happiness')
print('=' * 88)
print('\nTop confounders linked to both usage and happiness:')
print(confounder_audit[['Confounder', 'Corr with Usage', 'Corr with Happiness']].head(8).round(3).to_string(index=False))
print('\nNaive relationship:')
print(f'  Happiness change per extra Instagram minute = {naive_slope:+.4f}')
print(f'  Naive R^2 = {naive_r2:.3f}')
print('\nAdjusted relationship after controls:')
print(f'  Happiness change per extra Instagram minute = {adjusted_slope:+.4f}')
print(f'  Adjusted model R^2 = {adjusted_r2:.3f}')
print(f'  Partial usage-happiness correlation after controls = {partial_corr:+.3f}')
print('\nWithin-stress usage-happiness correlations:')
print(within_stress_corr.round(3).to_string(index=False))
print('\nAccount maturity comparison:')
print(account_maturity_summary.round(3).to_string(index=False))

## 9.2 Cohort Analysis by Account Creation Year

Here we treat users who joined Instagram in the same year as a cohort and compare how those cohorts differ in platform intensity, well-being, and audience size. This complements the causal section by showing whether earlier adopters and newer joiners follow different behavioral trajectories.

We summarize each cohort using average daily minutes, sessions per day, happiness, stress, followers, and user count, then visualize the cohort trends over account creation year.

In [ ]:
# Cohort analysis: compare users by the year they created their Instagram account.
cohort_analysis = (
    df_sample.groupby('account_creation_year', observed=True)
    .agg(
        avg_minutes=('daily_active_minutes_instagram', 'mean'),
        avg_sessions=('sessions_per_day', 'mean'),
        avg_happiness=('self_reported_happiness', 'mean'),
        avg_stress=('perceived_stress_score', 'mean'),
        avg_followers=('followers_count', 'mean'),
        user_count=('user_id', 'count'),
    )
    .round(2)
    .reset_index()
    .sort_values('account_creation_year')
)

reference_year = max(2026, int(cohort_analysis['account_creation_year'].max()))
cohort_analysis['cohort_age_years'] = reference_year - cohort_analysis['account_creation_year']
cohort_analysis['adopter_group'] = pd.cut(
    cohort_analysis['cohort_age_years'],
    bins=[-0.1, 3, 7, float('inf')],
    labels=['Recent Joiners', 'Mid-Tenure Cohorts', 'Early Adopters'],
)

print('Cohort Analysis by Account Creation Year:')
display(cohort_analysis)

fig = make_subplots(
    rows=2,
    cols=2,
    subplot_titles=[
        'Daily Instagram Minutes by Cohort',
        'Average Happiness by Cohort',
        'Average Stress by Cohort',
        'Average Followers by Cohort',
    ],
)

years = cohort_analysis['account_creation_year'].tolist()

fig.add_trace(
    go.Scatter(
        x=years,
        y=cohort_analysis['avg_minutes'],
        mode='lines+markers',
        marker_color=COLORS['primary'],
        name='Avg Minutes',
    ),
    row=1,
    col=1,
)
fig.add_trace(
    go.Scatter(
        x=years,
        y=cohort_analysis['avg_happiness'],
        mode='lines+markers',
        marker_color=COLORS['success'],
        name='Avg Happiness',
    ),
    row=1,
    col=2,
)
fig.add_trace(
    go.Scatter(
        x=years,
        y=cohort_analysis['avg_stress'],
        mode='lines+markers',
        marker_color=COLORS['secondary'],
        name='Avg Stress',
    ),
    row=2,
    col=1,
)
fig.add_trace(
    go.Scatter(
        x=years,
        y=cohort_analysis['avg_followers'],
        mode='lines+markers',
        marker_color=COLORS['accent'],
        name='Avg Followers',
    ),
    row=2,
    col=2,
)

fig.update_xaxes(title_text='Account creation year', row=1, col=1)
fig.update_xaxes(title_text='Account creation year', row=1, col=2)
fig.update_xaxes(title_text='Account creation year', row=2, col=1)
fig.update_xaxes(title_text='Account creation year', row=2, col=2)
fig.update_yaxes(title_text='Minutes / day', row=1, col=1)
fig.update_yaxes(title_text='Happiness score', row=1, col=2)
fig.update_yaxes(title_text='Stress score', row=2, col=1)
fig.update_yaxes(title_text='Followers', row=2, col=2)
fig.update_layout(height=650, showlegend=False, title_text='Cohort Analysis: Metrics by Account Creation Year')
fig.show()

cohort_group_summary = (
    cohort_analysis.groupby('adopter_group', observed=True)
    .agg(
        mean_minutes=('avg_minutes', 'mean'),
        mean_happiness=('avg_happiness', 'mean'),
        mean_stress=('avg_stress', 'mean'),
        mean_followers=('avg_followers', 'mean'),
        total_users=('user_count', 'sum'),
    )
    .round(2)
)

print('\nGrouped cohort summary:')
display(cohort_group_summary)

highest_usage_year = cohort_analysis.loc[cohort_analysis['avg_minutes'].idxmax(), 'account_creation_year']
lowest_happiness_year = cohort_analysis.loc[cohort_analysis['avg_happiness'].idxmin(), 'account_creation_year']
highest_followers_year = cohort_analysis.loc[cohort_analysis['avg_followers'].idxmax(), 'account_creation_year']

print('\nKey cohort takeaways:')
print(f'  Highest-usage cohort year      : {int(highest_usage_year)}')
print(f'  Lowest-happiness cohort year   : {int(lowest_happiness_year)}')
print(f'  Highest-follower cohort year   : {int(highest_followers_year)}')
print('  Interpret cohort gaps cautiously because the dataset is synthetic and not longitudinal.')

## 9.3 Geographic Patterns Analysis

This subsection compares Instagram behavior and well-being across countries, then contrasts urban and rural users. The goal is descriptive rather than causal: we want to see whether the platform-intensity patterns observed earlier are geographically uniform or whether some location-based structure appears in the synthetic data.

We summarize country-level averages for usage, happiness, stress, engagement, and sample size, keep only countries with strong representation, and then compare these national patterns with a simpler urban-versus-rural split.

In [ ]:
# Geographic analysis: compare country-level averages and the urban-rural split.
country_analysis = (
    df_sample.groupby('country', observed=True)
    .agg(
        avg_minutes=('daily_active_minutes_instagram', 'mean'),
        avg_happiness=('self_reported_happiness', 'mean'),
        avg_stress=('perceived_stress_score', 'mean'),
        avg_engagement=('user_engagement_score', 'mean'),
        user_count=('user_id', 'count'),
    )
    .round(2)
)
country_analysis = country_analysis[country_analysis['user_count'] >= 1000].sort_values('user_count', ascending=False)
country_analysis = country_analysis.reset_index()

urban_rural_analysis = (
    df_sample.groupby('urban_rural', observed=True)
    .agg(
        avg_minutes=('daily_active_minutes_instagram', 'mean'),
        avg_happiness=('self_reported_happiness', 'mean'),
        avg_stress=('perceived_stress_score', 'mean'),
        avg_sessions=('sessions_per_day', 'mean'),
        avg_engagement=('user_engagement_score', 'mean'),
        user_count=('user_id', 'count'),
    )
    .round(2)
    .reset_index()
)

print('Geographic Analysis (Countries with 1000+ users):')
display(country_analysis)
print('\nUrban vs Rural Comparison:')
display(urban_rural_analysis)

fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=['Average Daily Usage by Country', 'Happiness vs Stress by Country'],
)

fig.add_trace(
    go.Bar(
        x=country_analysis['country'],
        y=country_analysis['avg_minutes'],
        marker_color=COLORS['primary'],
        name='Avg Minutes',
        customdata=country_analysis[['user_count', 'avg_engagement']].to_numpy(),
        hovertemplate='<b>%{x}</b><br>Avg Minutes: %{y:.2f}<br>User Count: %{customdata[0]:,.0f}<br>Avg Engagement: %{customdata[1]:.2f}<extra></extra>',
    ),
    row=1,
    col=1,
)

fig.add_trace(
    go.Scatter(
        x=country_analysis['avg_stress'],
        y=country_analysis['avg_happiness'],
        mode='markers+text',
        text=country_analysis['country'],
        textposition='top center',
        marker=dict(
            size=(country_analysis['user_count'] / 400).clip(lower=10, upper=40),
            color=country_analysis['avg_minutes'],
            colorscale='Blues',
            showscale=True,
            colorbar=dict(title='Avg Minutes'),
            line=dict(width=1, color='white'),
            opacity=0.85,
        ),
        customdata=country_analysis[['avg_engagement', 'user_count']].to_numpy(),
        hovertemplate='<b>%{text}</b><br>Avg Stress: %{x:.2f}<br>Avg Happiness: %{y:.2f}<br>Avg Engagement: %{customdata[0]:.2f}<br>User Count: %{customdata[1]:,.0f}<extra></extra>',
        name='Country Pattern',
    ),
    row=1,
    col=2,
)

fig.update_layout(height=500, showlegend=False, title_text='Geographic Patterns')
fig.update_xaxes(title_text='Country', tickangle=45, row=1, col=1)
fig.update_yaxes(title_text='Average daily Instagram minutes', row=1, col=1)
fig.update_xaxes(title_text='Average stress', row=1, col=2)
fig.update_yaxes(title_text='Average happiness', row=1, col=2)
fig.show()

fig_urban = make_subplots(
    rows=1,
    cols=3,
    subplot_titles=['Usage Intensity', 'Happiness and Stress', 'Sessions and Engagement'],
)

fig_urban.add_trace(
    go.Bar(
        x=urban_rural_analysis['urban_rural'],
        y=urban_rural_analysis['avg_minutes'],
        marker_color=COLORS['secondary'],
        name='Avg Minutes',
    ),
    row=1,
    col=1,
)
fig_urban.add_trace(
    go.Bar(
        x=urban_rural_analysis['urban_rural'],
        y=urban_rural_analysis['avg_happiness'],
        marker_color=COLORS['success'],
        name='Avg Happiness',
    ),
    row=1,
    col=2,
)
fig_urban.add_trace(
    go.Bar(
        x=urban_rural_analysis['urban_rural'],
        y=urban_rural_analysis['avg_stress'],
        marker_color=COLORS['warning'],
        name='Avg Stress',
    ),
    row=1,
    col=2,
)
fig_urban.add_trace(
    go.Bar(
        x=urban_rural_analysis['urban_rural'],
        y=urban_rural_analysis['avg_sessions'],
        marker_color=COLORS['accent'],
        name='Avg Sessions',
    ),
    row=1,
    col=3,
)
fig_urban.add_trace(
    go.Scatter(
        x=urban_rural_analysis['urban_rural'],
        y=urban_rural_analysis['avg_engagement'],
        mode='lines+markers',
        marker_color=COLORS['info'],
        name='Avg Engagement',
        yaxis='y2',
    ),
    row=1,
    col=3,
)

fig_urban.update_layout(height=450, barmode='group', showlegend=True, title_text='Urban vs Rural Patterns')
fig_urban.update_xaxes(title_text='Area type', row=1, col=1)
fig_urban.update_xaxes(title_text='Area type', row=1, col=2)
fig_urban.update_xaxes(title_text='Area type', row=1, col=3)
fig_urban.update_yaxes(title_text='Average daily minutes', row=1, col=1)
fig_urban.update_yaxes(title_text='Average score', row=1, col=2)
fig_urban.update_yaxes(title_text='Average sessions', row=1, col=3)
fig_urban.show()

highest_usage_country = country_analysis.loc[country_analysis['avg_minutes'].idxmax(), 'country']
highest_happiness_country = country_analysis.loc[country_analysis['avg_happiness'].idxmax(), 'country']
highest_stress_country = country_analysis.loc[country_analysis['avg_stress'].idxmax(), 'country']

print('\nKey geographic takeaways:')
print(f'  Highest-usage country      : {highest_usage_country}')
print(f'  Highest-happiness country  : {highest_happiness_country}')
print(f'  Highest-stress country     : {highest_stress_country}')
print('  Cross-country gaps are modest, which is consistent with the synthetic and near-uniform nature of the dataset.')

# 10. Key Findings and Insights

This final section consolidates the main conclusions from the predictive modeling, user persona clustering, and advanced analyses.

## 10.1 Predictive Modeling Insights

1. **Perceived stress is the most credible predictive target.** The best regression model reached an $R^2$ of about **0.739**, which is strong but still believable for a synthetic behavioral dataset. This aligns with the earlier correlation analysis where Instagram activity variables were strongly and consistently associated with stress.
2. **Self-reported happiness is only weakly predictable.** The best happiness model achieved an $R^2$ of about **0.168**, meaning most variation in happiness remains unexplained by the available features. This suggests that happiness should be retained in the report, but framed as a weak and noisy target rather than a successful prediction task.
3. **User engagement score is suspiciously easy to predict.** The best engagement model achieved an $R^2$ near **0.999**, which is unusually high and strongly suggests that the target is mechanically tied to the same usage features used as predictors. This is useful descriptively, but should be interpreted as likely target-construction overlap rather than genuine real-world predictive power.
4. **Addiction-risk classification is near-perfect for the same reason.** Accuracy, F1, and ROC AUC were all extremely high, which is consistent with the fact that the binary target was derived from strongly overlapping usage-based features. The classification result is therefore informative about the dataset design, not necessarily about realistic detection performance.
5. **Feature importance confirms that Instagram intensity dominates the modeling section.** Across targets, variables such as `daily_active_minutes_instagram`, `usage_intensity_score`, `creator_score`, and session-based activity measures repeatedly emerged as the most influential predictors.

## 10.2 User Persona Clustering Insights

1. **The platform is best understood as a spectrum of usage intensity rather than health or lifestyle differences.** The clustering solution mainly separated users by Instagram behavior, engagement style, and stress-related intensity, while health-oriented features such as wellness remained almost constant across groups.
2. **The Doom Scroller and Social Butterfly represent the highest-risk personas.** These groups combine heavy usage with high stress and lower happiness, making them the clearest examples of problematic or compulsive platform interaction in this dataset.
3. **The Lurker appears to be the most psychologically favorable low-usage profile.** This persona shows relatively low usage, low stress, and the highest happiness, reinforcing the repeated pattern that lower platform intensity is associated with better well-being outcomes here.
4. **The Power User is structurally different from the other personas.** High follower counts combined with relatively low stress and high happiness suggest a more purposeful, efficient, or professional style of use rather than compulsive consumption.
5. **Clusters are informative but not fully natural.** The modest silhouette structure and the synthetic nature of the data mean the personas are analytically useful summaries, but they should not be treated as stable real-world social archetypes.

## 10.3 Advanced Analysis Insights

1. **The negative usage-happiness relationship remains after basic control adjustments.** In the causal-style section, the relationship between Instagram usage and happiness stayed negative even after controlling for age, stress, sleep, exercise, social activity, work hours, and socioeconomic context. This suggests the pattern is not explained away by the most obvious confounders available in the dataset.
2. **The stress-stratified analysis shows the same directional pattern.** Within lower-, middle-, and higher-stress groups, heavier usage still corresponded to lower average happiness, which strengthens the consistency of the observed relationship.
3. **Account-maturity and cohort analyses show stable behavior rather than radically different eras.** Newer cohorts generally used Instagram more and had more followers, but happiness and stress changed very little across cohort years. That means adoption era mainly affected platform intensity, not overall well-being outcomes.
4. **Geographic differences are weak.** Country-level and urban-rural comparisons showed only small differences in usage, happiness, stress, and engagement, which is exactly what we would expect from a highly synthetic and nearly uniform dataset.
5. **The advanced analyses support robustness, not causality.** Taken together, these checks make the negative usage-happiness pattern look internally consistent within this dataset, but they still do not justify a true causal claim because the data are synthetic, observational, and not generated from a real longitudinal process.

## 10.4 Overall Interpretation

1. **Instagram usage intensity is the central organizing signal in the entire project.** It drives the strongest predictive results, defines the main user personas, and remains the key variable in the advanced analyses.
2. **Stress is the clearest and most defensible outcome variable.** It is strongly associated with platform activity, reasonably predictable, and consistent across EDA, modeling, and clustering.
3. **Happiness is substantively interesting but analytically weak.** It should remain in the report because it contributes an important interpretive dimension, but the notebook should explicitly note that it is much harder to predict than stress.
4. **Some results are informative partly because they reveal how the synthetic dataset was constructed.** Extremely strong performance for engagement and addiction-risk prediction, weak geographic differences, and nearly flat health variation all point to structural simplifications in the data-generation process.
5. **The project’s strongest conclusion is not that Instagram literally causes lower happiness, but that heavier Instagram use is consistently associated with higher stress, lower happiness, and more high-risk behavioral profiles throughout this dataset.**